# Coco Crepe — 07 Gold Inventory Status

Publicación del Data Product final de inventario.

In [0]:
GROUP = "g203"

SALES_DATA_PRODUCT = "sales_summary"
INVENTORY_DATA_PRODUCT = "inventory_status"
PRODUCT_DATA_PRODUCT = "product_master"

# Completa estos valores solo si la detección automática no encuentra
# exactamente un catálogo por Data Product.
SALES_CATALOG_MANUAL = None
INVENTORY_CATALOG_MANUAL = "g203_inv_inventory_status"
PRODUCT_CATALOG_MANUAL = None

def resolve_catalog(data_product_name, manual_catalog=None):
    if manual_catalog:
        return manual_catalog

    catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
    target = data_product_name.lower()

    preferred = [
        catalog for catalog in catalogs
        if GROUP.lower() in catalog.lower()
        and target in catalog.lower()
    ]

    if len(preferred) == 1:
        return preferred[0]

    matches = [
        catalog for catalog in catalogs
        if target in catalog.lower()
    ]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(
        f"No se pudo identificar un catálogo único para '{data_product_name}'. "
        f"Catálogos visibles: {catalogs}. "
        "Completa la variable *_CATALOG_MANUAL correspondiente."
    )

SALES_CATALOG = resolve_catalog(
    SALES_DATA_PRODUCT,
    SALES_CATALOG_MANUAL
)

INVENTORY_CATALOG = resolve_catalog(
    INVENTORY_DATA_PRODUCT,
    INVENTORY_CATALOG_MANUAL
)

PRODUCT_CATALOG = resolve_catalog(
    PRODUCT_DATA_PRODUCT,
    PRODUCT_CATALOG_MANUAL
)

print(f"Sales catalog: {SALES_CATALOG}")
print(f"Inventory catalog: {INVENTORY_CATALOG}")
print(f"Product catalog: {PRODUCT_CATALOG}")

In [0]:
inventory_detail = f"{INVENTORY_CATALOG}.silver.inventory_detail"
inventory_status = f"{INVENTORY_CATALOG}.gold.inventory_status" 

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {inventory_status} AS
SELECT
    product_id,
    product_name,
    category,
    stock,
    CASE
        WHEN stock < 50 THEN 'LOW'
        WHEN stock <= 150 THEN 'MEDIUM'
        ELSE 'OK'
    END AS stock_status,
    product_price,
    ROUND(stock * product_price, 2) AS inventory_value,
    supplier_id,
    supplier_name,
    supplier_city,
    last_update,
    current_timestamp() AS published_at
FROM {inventory_detail}
""")

In [0]:
spark.sql(f"""
SELECT
    COUNT(*) AS record_count,
    SUM(CASE WHEN stock_status = 'LOW' THEN 1 ELSE 0 END) AS low_stock_products,
    SUM(CASE WHEN stock_status = 'MEDIUM' THEN 1 ELSE 0 END) AS medium_stock_products,
    SUM(CASE WHEN stock_status = 'OK' THEN 1 ELSE 0 END) AS ok_stock_products,
    ROUND(SUM(inventory_value), 2) AS total_inventory_value
FROM {inventory_status}
""").display()

spark.sql(f"""
    SELECT *
    FROM {inventory_status}
    ORDER BY
        CASE stock_status
            WHEN 'LOW' THEN 1
            WHEN 'MEDIUM' THEN 2
            ELSE 3
        END,
        stock ASC
    LIMIT 30
""").display()